# CropGuard — train the disease model

Trains the disease branch on **Plant-Diseases-100k-Labelled-Images** and exports
a bundle that runs on a field device with no internet.

**Runtime → Change runtime type → T4 GPU** before starting. ~100k images at 224px
is roughly 1.5–3 hours for 25 epochs on a T4; start with fewer epochs if you just
want to see it work.

The pest branch is a separate notebook (`train_pest_colab.ipynb`) and a separate
model. They are kept apart deliberately: the two corpora disagree about what a
class means, and a single model trained on both would have the same condition
labelled from two sources.


## 1. Check the GPU


In [ ]:
import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('NO GPU — Runtime > Change runtime type > T4 GPU, then rerun.')
    print('100k images on CPU is not worth attempting.')


## 2. Get the code


In [ ]:
!git clone -b claude/smart-farming-edge-ai-2hzch8 https://github.com/omsatpute61-afk/Model.git
%cd Model
!git log --oneline -1


### Dependencies

Colab's torch is already built against its CUDA driver — **do not reinstall it**.


In [ ]:
!pip install -q kagglehub onnx onnxruntime pyyaml

import sys
sys.path.insert(0, '/content/Model/src')

import cropguard
from cropguard.taxonomy import load_taxonomy
print('cropguard', cropguard.__version__, '|', len(load_taxonomy()), 'taxonomy classes')


## 3. Download the corpus

~100k images, so expect a few minutes. If you hit a 401/403, add `KAGGLE_USERNAME`
and `KAGGLE_KEY` via the Colab sidebar → 🔑 Secrets and rerun.


In [ ]:
import os
try:
    from google.colab import userdata
    for key in ('KAGGLE_USERNAME', 'KAGGLE_KEY'):
        try:
            os.environ[key] = userdata.get(key)
        except Exception:
            pass
except ImportError:
    pass

import kagglehub
DISEASE_PATH = kagglehub.dataset_download('salmasyed1360/plant-diseases-100k-labelled-images')
print('downloaded to:', DISEASE_PATH)


### What is in it, and what will map

The taxonomy carries PlantVillage-style aliases, so most folders should resolve.
Anything unmapped is a class worth adding — send me the list.


In [ ]:
from pathlib import Path
from collections import Counter
from cropguard.data.ingest import find_dataset_root, looks_like_disease

IMG = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}
root = find_dataset_root(DISEASE_PATH)   # same helper the ingest uses
print('dataset root:', root, '\n')

counts = Counter()
for d in sorted(p for p in root.iterdir() if p.is_dir()):
    counts[d.name] = sum(1 for f in d.rglob('*') if f.suffix.lower() in IMG)
print(f'{len(counts)} class folders, {sum(counts.values()):,} images\n')

tax = load_taxonomy()
mapped, unmapped = [], []
for name, n in counts.items():
    cls = tax.resolve(name)
    (mapped if cls else unmapped).append((name, n, cls.id if cls else None))

print(f'MAPPED: {len(mapped)} classes, {sum(n for _, n, _ in mapped):,} images')
for name, n, cid in sorted(mapped, key=lambda x: -x[1])[:40]:
    print(f'   {n:>7,}  {name}  ->  {cid}')

print(f'\nUNMAPPED: {len(unmapped)} classes, {sum(n for _, n, _ in unmapped):,} images')
for name, n, _ in sorted(unmapped, key=lambda x: -x[1]):
    print(f'   {n:>7,}  {name}')
print('\nSend the UNMAPPED list back — those need taxonomy entries or aliases,')
print('and until they have them the model simply cannot diagnose them.')


## 4. Ingest → clean → split → EDA

Declared as a **disease** source, so any insect class found in it is rejected —
the enforcement runs both ways. Cleaning removes unreadable files, duplicates
(including the same image under two labels, which caps achievable accuracy) and
near-duplicates that would otherwise leak across the split.

On 100k images the inspection pass reads every file once; allow a few minutes.


In [ ]:
!python scripts/prepare_real_dataset.py \
    --disease "$DISEASE_PATH" \
    --out artifacts/data/disease \
    --min-per-class 40 \
    --workers 2


### Read the EDA before committing to a long run

Its recommendations name the settings in `configs/disease.yaml` to change. On a
corpus this size, getting `image_size` and `use_focal` right before a 2-hour run
is worth the five minutes of reading.


In [ ]:
print(open('artifacts/data/disease/eda.md').read())


## 5. Train

Apply anything the EDA flagged first. A quick sanity run before the real one is
cheap insurance:

```
!python -m cropguard.train --config configs/disease.yaml --set optim.epochs=2
```


In [ ]:
!python -m cropguard.train --config configs/disease.yaml


## 6. Evaluate

Macro F1, not accuracy — a corpus with large healthy classes gives a flattering
accuracy to a model that misses every rare disease. Also reports the
**cross-category error rate**: confusing a disease with a deficiency sends the
farmer to buy the wrong input.


In [ ]:
!python -m cropguard.evaluate --run artifacts/runs/cropguard-disease --split test


## 7. Export the edge bundle


In [ ]:
!python -m cropguard.export --run artifacts/runs/cropguard-disease --formats onnx,int8
!echo '--- bundle ---' && ls -la artifacts/runs/cropguard-disease/export/


In [ ]:
!python -m cropguard.benchmark --bundle artifacts/runs/cropguard-disease/export --compare


## 8. Try the device runtime

Image in, calibrated diagnosis out, plus the advisory a farmer receives — and the
decision to refuse an image the model does not understand.


In [ ]:
import random, numpy as np
from PIL import Image
from cropguard.edge import EdgeClassifier
from cropguard.data.manifest import read_manifest

clf = EdgeClassifier('artifacts/runs/cropguard-disease/export')
print('backend:', clf.backend, '| model:', clf._model_path.name)
print('threshold:', clf.card.policy.min_confidence, '| classes:', len(clf.card.class_ids))

records = [r for r in read_manifest('artifacts/data/disease/manifest.csv') if r.split == 'test']
random.seed(0)
correct = 0
for rec in random.sample(records, min(8, len(records))):
    d = clf.diagnose(rec.path)
    correct += d.class_id == rec.class_id
    mark = 'ok ' if d.class_id == rec.class_id else '   '
    print(f'{mark}{rec.class_id:<30} -> {d.class_id:<30} p={d.confidence:.2f} {d.latency_ms:5.1f}ms')
    if d.accepted and d.advisory and d.advisory.urgency in ('warning', 'critical'):
        print(f'      SMS: {d.advisory.to_sms()}')
print(f'\n{correct}/8 correct on this sample')

print('\n--- an image the model should refuse ---')
noise = Image.fromarray(np.random.default_rng(0).integers(0, 255, (320, 320, 3)).astype('uint8'))
d = clf.diagnose(noise)
print('accepted:', d.accepted, '| confidence:', round(d.confidence, 3),
      '| novelty:', None if d.novelty is None else round(d.novelty))
print('reason:', d.reason or '(accepted)')


## 9. Download the bundle


In [ ]:
import shutil
from google.colab import files

shutil.make_archive('cropguard_disease_bundle', 'zip', 'artifacts/runs/cropguard-disease/export')
files.download('cropguard_disease_bundle.zip')


---

### If accuracy is short

1. **Read `eda.md` again** — usually the answer is there.
2. `--set data.image_size=160` if the EDA says most images are small.
3. `--set optim.use_focal=true` if the imbalance ratio is above ~20:1.
4. `--set optim.epochs=40` — 25 is conservative.
5. `--set model.backbone=efficientnet_b0` if the device budget allows, then
   re-benchmark on the real board.

### The number that actually matters

PlantVillage-derived corpora are studio-lit, single leaf, plain background. Models
score in the high nineties on them and then lose 30–40 points on a phone photo
taken in a standing crop. **Hold back a few hundred genuinely field-shot images
as a second test set.** That score is the one worth reporting, and the one that
predicts whether this helps a farmer.
